# AAI614 – FluWatch Web Scraping

This notebook scrapes Table 2 from the FluWatch respiratory virus surveillance webpage and creates a pandas DataFrame with one row for each province or territory.

In [12]:
from urllib.request import urlopen
from bs4 import BeautifulSoup
import pandas as pd
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

url = 'https://www.canada.ca/en/public-health/services/surveillance/respiratory-virus-detections-canada/2018-2019/respiratory-virus-detections-isolations-week-01-ending-january-5-2019.html'

html = urlopen(url)
bs = BeautifulSoup(html, 'html.parser')

print(bs.title)

<title>Respiratory Virus Report, Week 01 ending  January 5, 2019 - Canada.ca</title>


In [13]:
tables = bs.find_all('table')

print("Number of tables found:", len(tables))

Number of tables found: 10


In [14]:
for i, table in enumerate(tables):
    print("TABLE", i)
    print(table.get_text(strip=True)[:200])
    print()

TABLE 0
Table 1: Respiratory Virus Detections/Isolations for the week ending January 5, 2019 (Reporting Week 201901)Reporting LaboratoryFlu TestedA(H1N1)pdm09 PositiveA(H3) PositiveA(UnS) PositiveTotal Flu A 

TABLE 1
Table 2: Respiratory Virus Detections/Isolations for the period August 26, 2018 - January 5, 2019 (Reporting Weeks 201835-201901)Reporting LaboratoryFlu TestedA(H1N1)pdm09 PositiveA(H3) PositiveA(UnS)

TABLE 2
Number positive laboratory tests for other respiratory viruses by report week, Canada, 2018-19WeekParaInfluenzaAdenovirusHuman metapneumovirusEnterovirus/RhinovirusCoronavirusRespiratory syncytial vir

TABLE 3
Positive Influenza Tests (%) in Canada by Region by Week of ReportWeekWeek endCan TestsCan A%Can B%Atl TestsAtl A%Atl B%QC TestsQC A%QC B%ON TestsON A%ON B%Pr TestsPr A%Pr B%BC TestsBC A%BC B%Terr Tes

TABLE 4
Positive Respiratory syncytial virus (RSV) Tests (%) in Canada by Region by Week of ReportWeekWeek endCan TestsRSV%Atl TestsRSV%QC TestsRSV%ON TestsRSV%

In [15]:
table2 = tables[1]

rows = table2.find_all('tr')

print("Number of rows:", len(rows))

Number of rows: 43


In [16]:
for row in rows[:5]:
    cells = row.find_all(['th', 'td'])
    data = [cell.get_text(strip=True) for cell in cells]
    print(data)

['Reporting Laboratory', 'Flu Tested', 'A(H1N1)pdm09 Positive', 'A(H3) Positive', 'A(UnS) Positive', 'Total Flu A Positive', 'Total Flu B Positive', 'RSV Tested', 'RSV Positive', 'PIV Tested', 'PIV 1 Positive', 'PIV 2 Positive', 'PIV 3 Positive', 'PIV 4 Positive', 'Other PIV Positive', 'Adeno Tested', 'Adeno Positive', 'hMPV Tested', 'hMPV Positive', 'Entero/Rhino Tested', 'Entero/Rhino Positive', 'Coron Tested', 'Coron Positive']
['Newfoundland', '1299', '1', '0', '113', '114', '1', '1299', '91', '1299', '0', '10', '34', '0', '0', '1299', '12', '1299', '8', '1299', '200', 'N.A.', 'N.A.']
['Prince Edward Island', '307', '38', '0', '0', '38', '0', '305', '5', '53', '0', '0', '1', '3', '0', '48', '5', '48', '0', '48', '21', '48', '0']
['Nova Scotia', '864', '0', '0', '52', '52', '1', '869', '45', '322', '0', '3', '7', '1', '0', '322', '0', '322', '3', '322', '53', '322', '1']
['New Brunswick', '4271', '42', '1', '715', '758', '2', '4274', '131', '1185', '0', '12', '6', '29', '0', '1185',

In [17]:
for row in rows[1:]:
    cells = row.find_all(['th', 'td'])
    if cells:
        print(cells[0].get_text(strip=True))

Newfoundland
Prince Edward Island
Nova Scotia
New Brunswick
Atlantic
Région    Nord-Est
Québec-Chaudière-Appalaches
Centre-du-Québec
Montréal-Laval
Ouest du Québec
Montérégie
Province of Québec
Ottawa P.H.L.
CHEO - Ottawa
Kingston P.H.L.
UHN / Mount Sinai Hospital
P.H.O.L. - Toronto
Sick Kids Hospital - Toronto
Sunnybrook & Women's    College HSC
Sault Ste. Marie P.H.L.
Timmins P.H.L.
St. Joseph's - London
London P.H.L.
Orillia P.H.L.
Thunder Bay P.H.L.
Sudbury P.H.L.
Hamilton P.H.L.
Peterborough    P.H.L.
Province of Ontario
Manitoba
Regina
Saskatoon
Province of    Saskatchewan
Province of Alberta
Prairies
British    Columbia
Yukon
Northwest    Territories
Nunavut
Territories
CANADA
Specimens from YT, NT and NU are sent to reference laboratories in other provinces and reported results reflect specimens identified as originating from YT, NT or NU.Delays in the reporting of data may cause data to change retrospectively.Due to reporting delays, the sum of weekly report totals do not add 

In [18]:
headers = [cell.get_text(strip=True) for cell in rows[0].find_all(['th', 'td'])]

data = []

for row in rows[1:]:
    cells = row.find_all(['th', 'td'])
    values = [cell.get_text(strip=True) for cell in cells]

    if len(values) == len(headers):
        data.append(values)

df = pd.DataFrame(data, columns=headers)

df.head()

,Reporting Laboratory,Flu Tested,A(H1N1)pdm09 Positive,A(H3) Positive,A(UnS) Positive,Total Flu A Positive,Total Flu B Positive,RSV Tested,RSV Positive,PIV Tested,...,PIV 4 Positive,Other PIV Positive,Adeno Tested,Adeno Positive,hMPV Tested,hMPV Positive,Entero/Rhino Tested,Entero/Rhino Positive,Coron Tested,Coron Positive
0,Newfoundland,1299,1,0,113,114,1,1299,91,1299,...,0,0,1299,12,1299,8,1299,200,N.A.,N.A.
1,Prince Edward Island,307,38,0,0,38,0,305,5,53,...,3,0,48,5,48,0,48,21,48,0
2,Nova Scotia,864,0,0,52,52,1,869,45,322,...,1,0,322,0,322,3,322,53,322,1
3,New Brunswick,4271,42,1,715,758,2,4274,131,1185,...,29,0,1185,84,1185,7,1185,201,1185,6
4,Atlantic,6741,81,1,880,962,4,6747,272,2859,...,33,0,2854,101,2854,18,2854,475,1555,7


In [19]:
provinces_territories = [
    'Newfoundland',
    'Prince Edward Island',
    'Nova Scotia',
    'New Brunswick',
    'Province of Québec',
    'Province of Ontario',
    'Manitoba',
    'Province of Saskatchewan',
    'Province of Alberta',
    'British Columbia',
    'Yukon',
    'Northwest Territories',
    'Nunavut']

final_df = df[df['Reporting Laboratory'].isin(provinces_territories)]

final_df

,Reporting Laboratory,Flu Tested,A(H1N1)pdm09 Positive,A(H3) Positive,A(UnS) Positive,Total Flu A Positive,Total Flu B Positive,RSV Tested,RSV Positive,PIV Tested,...,PIV 4 Positive,Other PIV Positive,Adeno Tested,Adeno Positive,hMPV Tested,hMPV Positive,Entero/Rhino Tested,Entero/Rhino Positive,Coron Tested,Coron Positive
0,Newfoundland,1299,1,0,113,114,1,1299,91,1299,...,0,0,1299,12,1299,8,1299,200,N.A.,N.A.
1,Prince Edward Island,307,38,0,0,38,0,305,5,53,...,3,0,48,5,48,0,48,21,48,0
2,Nova Scotia,864,0,0,52,52,1,869,45,322,...,1,0,322,0,322,3,322,53,322,1
3,New Brunswick,4271,42,1,715,758,2,4274,131,1185,...,29,0,1185,84,1185,7,1185,201,1185,6
11,Province of Québec,38395,0,0,5974,5975,81,35755,2816,11201,...,36,0,11288,373,10786,63,N.A.,N.A.,10703,310
28,Province of Ontario,23795,752,180,546,1478,42,23518,1123,13600,...,38,0,13348,117,13307,101,3652,683,3080,117
29,Manitoba,4911,192,3,486,681,3,4828,101,2254,...,10,0,2254,25,1214,9,2254,310,1248,12
33,Province of Alberta,18824,3132,73,1540,4745,29,15040,0,15040,...,154,0,15040,184,15040,189,15040,2693,15040,321
36,Yukon,143,16,1,6,23,0,122,2,N.A.,...,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.
38,Nunavut,33,8,0,0,8,0,33,0,33,...,0,0,33,5,32,0,33,10,32,0


In [21]:
final_df = df.loc[[0, 1, 2, 3, 11, 28, 29, 32, 33, 35, 36, 37, 38]]

final_df

,Reporting Laboratory,Flu Tested,A(H1N1)pdm09 Positive,A(H3) Positive,A(UnS) Positive,Total Flu A Positive,Total Flu B Positive,RSV Tested,RSV Positive,PIV Tested,...,PIV 4 Positive,Other PIV Positive,Adeno Tested,Adeno Positive,hMPV Tested,hMPV Positive,Entero/Rhino Tested,Entero/Rhino Positive,Coron Tested,Coron Positive
0,Newfoundland,1299,1,0,113,114,1,1299,91,1299,...,0,0,1299,12,1299,8,1299,200,N.A.,N.A.
1,Prince Edward Island,307,38,0,0,38,0,305,5,53,...,3,0,48,5,48,0,48,21,48,0
2,Nova Scotia,864,0,0,52,52,1,869,45,322,...,1,0,322,0,322,3,322,53,322,1
3,New Brunswick,4271,42,1,715,758,2,4274,131,1185,...,29,0,1185,84,1185,7,1185,201,1185,6
11,Province of Québec,38395,0,0,5974,5975,81,35755,2816,11201,...,36,0,11288,373,10786,63,N.A.,N.A.,10703,310
28,Province of Ontario,23795,752,180,546,1478,42,23518,1123,13600,...,38,0,13348,117,13307,101,3652,683,3080,117
29,Manitoba,4911,192,3,486,681,3,4828,101,2254,...,10,0,2254,25,1214,9,2254,310,1248,12
32,Province of Saskatchewan,8391,1280,60,749,2089,3,8391,160,10324,...,40,0,8391,106,8391,47,8391,1337,8391,30
33,Province of Alberta,18824,3132,73,1540,4745,29,15040,0,15040,...,154,0,15040,184,15040,189,15040,2693,15040,321
35,British Columbia,8731,1010,75,755,1840,10,8731,279,4342,...,66,0,4342,56,4342,14,3442,982,3442,54


## Conclusion
Table 2 was scraped from the FluWatch webpage and organized into a pandas DataFrame. The final DataFrame contains one row for each province or territory in Canada.